In [5]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# ===================== 1. 读取数据 =====================
# 读取成分-靶点数据
df_compound = pd.read_excel("成分-靶点列表.xlsx")
df_compound.columns = ["compound", "target_gene"]

# 读取哮喘表达数据
expr = pd.read_csv(
    "GSE63142_series_matrix.txt",
    sep="\t",
    comment="!",
    index_col=0
)
expr = np.log2(expr + 1)

# ===================== 2. 样本分组 =====================
groups = []
with open("GSE63142_series_matrix.txt", "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("!Sample_characteristics_ch1"):
            parts = line.strip().split("\t")[1:]
            for p in parts:
                if "control" in p:
                    groups.append("Control")
                elif "not severe" in p:
                    groups.append("Mild_Asthma")
                elif "severe" in p:
                    groups.append("Severe_Asthma")
            break

# ===================== 3. 直接用哮喘经典通路来筛选成分=====================
# 哮喘核心通路的关键基因列表
asthma_pathway_genes = [
    "TNF", "IL17A", "IL17F", "NFKB1", "NFKB2", "RELA", "TNFAIP3", "TNFSF13B",
    "CXCL8", "CXCL1", "CXCL2", "CCL2", "CCL5", "IL6", "IL1B", "IFNG",
    "IL4", "IL5", "IL13", "IL10", "TGFB1", "VEGFA", "MMP9", "PTGS2"
]

# 统一基因名格式，忽略大小写和空格
asthma_pathway_genes_clean = [g.upper().strip() for g in asthma_pathway_genes]
df_compound["target_gene_clean"] = df_compound["target_gene"].str.upper().str.strip()

# 筛选出靶点属于哮喘通路的成分
df_common = df_compound[df_compound["target_gene_clean"].isin(asthma_pathway_genes_clean)]

# 统计每个成分对应多少个哮喘通路基因
compound_count = df_common.groupby("compound")["target_gene"].nunique().sort_values(ascending=False).reset_index()
compound_count.columns = ["compound", "asthma_target_count"]

print("\n===== 治疗哮喘的潜在核心成分 TOP20 =====")
print(compound_count.head(20))

# ===================== 4. 保存到 Middle 文件夹 =====================
middle_dir = "../Middle"
os.makedirs(middle_dir, exist_ok=True)

compound_count.to_excel("../Middle/哮喘_核心成分筛选结果.xlsx", index=False)
df_common[["compound", "target_gene"]].to_excel("../Middle/哮喘_成分-靶点对应关系.xlsx", index=False)

print(f"\n文件已成功保存到 Middle 文件夹！")
print(f"保存路径：{os.path.abspath('../Middle')}")
print("1. 哮喘_核心成分筛选结果.xlsx：按抗哮喘潜力排序的核心成分列表")
print("2. 哮喘_成分-靶点对应关系.xlsx：成分-哮喘通路靶点的一一对应明细")


===== 治疗哮喘的潜在核心成分 TOP20 =====
                   compound  asthma_target_count
0                Albiflorin                    2
1                   Icariin                    2
2                Epimedin C                    2
3         Glycyrrhetic acid                    2
4                Epimedin A                    2
5                Epimedin B                    2
6            Liquiritigenin                    2
7       Benzoylpaeoniflorin                    1
8          Astragaloside IV                    1
9    5-O-Methylvisammioside                    1
10   4-Hydroxycinnamic acid                    1
11            Baohuoside I                     1
12  Calycosin 7-O-glucoside                    1
13     Desbenzoylalbiflorin                    1
14        Isoliquiritigenin                    1
15           Licochalcone B                    1
16                   Ononin                    1
17          Oxypaeoniflorin                    1
18             Paeoniflorin           

In [ ]:
import pandas as pd

# 1. 读取数据
# 成分-靶点对应关系
compound_target = pd.read_excel("../Middle/哮喘_成分-靶点对应关系.xlsx")
# 哮喘患者差异表达基因
# 要求：差异基因文件必须有一列是基因名
degs = pd.read_excel("../Middle/哮喘患者差异表达基因.xlsx")

# 2. 提取药物靶点基因列表
drug_targets = compound_target["target_gene"].unique().tolist()
# 提取哮喘差异基因列表
asthma_degs = degs["gene_name"].unique().tolist()

# 3. 取交集：药物直接作用的哮喘核心靶点
core_targets = list(set(drug_targets) & set(asthma_degs))
print(f"益气温阳护卫汤治疗哮喘的核心靶点共 {len(core_targets)} 个：")
print(core_targets)

# 4. 输出核心靶点对应的成分-靶点关系
core_compound_target = compound_target[compound_target["target_gene"].isin(core_targets)]
core_compound_target.to_excel("../Middle/核心成分-靶点对应关系.xlsx", index=False)
print("核心成分-靶点对应关系已保存到 /mnt/核心成分-靶点对应关系.xlsx")